# **Part A — Standup**

**DONE :-** Day 41 project scope, dataset selection, initial data checks, evaluation metric (Average Precision), and a provisional GOOD ENOUGH target of AP ≥ 0.20 have been documented.  
**BLOCKED :-** The previously reported test positive rate of 0.3158 conflicts with the expected target distribution. I need to verify the target labels and evaluation split before trusting the baseline score.  
**TODAY :-** Validate the split, implement a strong Logistic Regression baseline, train one untuned Pipeline model, compare scores, measure evaluation variability, document the gap, and commit the results.

# **Part B — Make your baseline non-trivial**

In [8]:
# Import libraries and load the dataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

df = pd.read_csv("/home/ctpc-29/bank-marketing-capstone/data/bank-full.csv", sep=";")

print("Dataset Shape:", df.shape)

print("\nTarget Distribution:")
print(df["y"].value_counts())

print("\nTarget Proportions:")
print(df["y"].value_counts(normalize=True))

print("\nMissing Values:")
print(df.isnull().sum().sum())

Dataset Shape: (45211, 17)

Target Distribution:
y
no     39922
yes     5289
Name: count, dtype: int64

Target Proportions:
y
no     0.883015
yes    0.116985
Name: proportion, dtype: float64

Missing Values:
0


In [12]:
# Check the target and split
print("Target Values:", df["y"].unique())
print("Total positive customers =", (df["y"] == "yes").sum())
print("Overall positive rate =", round((df["y"] == "yes").mean(), 3))

Target Values: <StringArray>
['no', 'yes']
Length: 2, dtype: str
Total positive customers = 5289
Overall positive rate = 0.117


In [13]:
# Create a verified stratified split
X = df.drop(columns=["y"])
y = (df["y"] == "yes").astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print("Training Rows:", X_train.shape[0])
print("Testing Rows:", X_test.shape[0])

print("\nTraining positive rate:", round(y_train.mean(), 4))
print("Testing positive rate:", round(y_test.mean(), 4))
print("Overall positive rate:", round(y.mean(), 4))

Training Rows: 36168
Testing Rows: 9043

Training positive rate: 0.117
Testing positive rate: 0.117
Overall positive rate: 0.117


In [14]:
# Create the trivial baseline
trivial_model = DummyClassifier(strategy="prior")
trivial_model.fit(X_train, y_train)
trivial_scores = trivial_model.predict_proba(X_test)[:, 1]
trivial_ap = average_precision_score(y_test, trivial_scores)

print("Trivial Baseline AP:", round(trivial_ap, 4))

Trivial Baseline AP: 0.117


In [15]:
# Prepare preprocessing for real models
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categoric_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numeric Features:", numeric_features) 
print("Categorical Features:", categoric_features)

Numeric Features: ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
Categorical Features: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


In [19]:
numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_preprocessor, numeric_features),
    ("categorical", categorical_preprocessor, categoric_features)
])

In [20]:
# Build the strong baseline
strong_features = ["age", "job", "marital", "education", "housing", "loan"]

X_train_strong = X_train[strong_features]
X_test_strong = X_test[strong_features]

strong_numeric = ["age"]
strong_categorical = ["job", "marital", "education", "housing", "loan"]

strong_preprocessor = ColumnTransformer([
    ("numeric", numeric_preprocessor, strong_numeric),
    ("categorical", categorical_preprocessor, strong_categorical)
])

strong_baseline = Pipeline([
    ("preprocessing", strong_preprocessor),
    ("model", LogisticRegression())
])
strong_baseline.fit(X_train_strong, y_train)

strong_scores = strong_baseline.predict_proba(X_test_strong)[:, 1]
strong_ap = average_precision_score(y_test, strong_scores)

print("Strong baseline AP:", round(strong_ap, 4))

Strong baseline AP: 0.2142


# **Part C — Train your first real model**

In [21]:
# Train a Random Forest inside a Pipeline
first_model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])
first_model.fit(X_train, y_train)

model_scores = first_model.predict_proba(X_test)[:, 1]
model_ap = average_precision_score(y_test, model_scores)

print("First Real Model: Random Forest")
print("First Model AP:", round(model_ap, 4))

First Real Model: Random Forest
First Model AP: 0.6104


In [22]:
# Build the honest comparison table
good_enough_target = 0.20

results = pd.DataFrame({
    "What": ["Trivial Baseline", "Strong Baseline - Logistic Regression",
             "First Real Model - Random Forest", "Good Enough Target"],
    "AP Scores": [trivial_ap, strong_ap, model_ap, good_enough_target],
})

results["VS Strong Baseline"] = [trivial_ap - strong_ap, 0.0, model_ap - strong_ap,
                                good_enough_target - strong_ap]

print(results.round(4).to_string(index=False))

                                 What  AP Scores  VS Strong Baseline
                     Trivial Baseline     0.1170             -0.0972
Strong Baseline - Logistic Regression     0.2142              0.0000
     First Real Model - Random Forest     0.6104              0.3962
                   Good Enough Target     0.2000             -0.0142


* A positive Random Forest difference means it beat Logistic Regression.
* A negative difference means the strong baseline won.
* A target difference of 0.0000 means the strong baseline exactly meets the target.
* A target difference below zero means the strong baseline already exceeds the target.

In [25]:
# Check score variability across five splits
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(first_model, X_train, y_train, cv=cv, scoring="average_precision")

print("Cross-Validation AP Scores:")
print(np.round(cv_scores, 4))

print("\nLoweset AP   :", round(cv_scores.min(), 4))
print("Highest AP    :", round(cv_scores.max(), 4))
print("Mean AP       :", round(cv_scores.mean(), 4))
print("Score Spread  :", round(cv_scores.max() - cv_scores.min(), 4))
print("Test AP       :", round(model_ap, 4))
print("Difference From Strong baseline:", round(model_ap - strong_ap, 4))

Cross-Validation AP Scores:
[0.5789 0.608  0.621  0.6179 0.6137]

Loweset AP   : 0.5789
Highest AP    : 0.621
Mean AP       : 0.6079
Score Spread  : 0.042
Test AP       : 0.6104
Difference From Strong baseline: 0.3962


In [26]:
results.to_csv("Day42_model_comparison.csv", index=False)

print("Comparison Table saved successfully")

Comparison Table saved successfully


In [33]:
# Evaluate both models across three splits
from sklearn.base import clone
split_results = []
random_states = [42, 100, 2026]

for seed in random_states:
    X_train_split, X_test_split, y_train_split, y_test_split = (train_test_split(
        X, y, test_size=0.20, random_state=seed, stratify=y
    ))

    # Strong Baseline: Logistic Regression
    strong_model_split = clone(strong_baseline)
    strong_model_split.fit(X_train_split[strong_features], y_train_split)
    strong_predictions = strong_model_split.predict_proba(X_test_split[strong_features])[:, 1]
    strong_ap_split = average_precision_score(y_test_split, strong_predictions)

    # First Real Model: Random Forest
    rf_model_split = clone(first_model)
    rf_model_split.fit(X_train_split, y_train_split)
    rf_predictions = rf_model_split.predict_proba(X_test_split)[:, 1]
    rf_ap_split = average_precision_score(y_test_split, rf_predictions)

    split_results.append({
        "Random State": seed,
        "Strong Baseline AP": strong_ap_split,
        "Random Forest AP": rf_ap_split,
        "Gain vs Strong Baseline": rf_ap_split - strong_ap_split
    })


split_results_df = pd.DataFrame(split_results)

print(split_results_df.round(4).to_string(index=False))

 Random State  Strong Baseline AP  Random Forest AP  Gain vs Strong Baseline
           42              0.2142            0.6104                   0.3962
          100              0.2240            0.6157                   0.3917
         2026              0.2254            0.6113                   0.3859


In [31]:
# Calculate the spread
strong_low = split_results_df["Strong Baseline AP"].min()
strong_high = split_results_df["Strong Baseline AP"].max()

rf_low = split_results_df["Random Forest AP"].min()
rf_high = split_results_df["Random Forest AP"].max()

gain_low = split_results_df["Gain vs Strong Baseline"].min()
gain_high = split_results_df["Gain vs Strong Baseline"].max()

print("Strong Baseline")
print("-" * 17)
print("Lowest AP:", round(strong_low, 4))
print("Highest AP:", round(strong_high, 4))
print("Spread:", round(strong_high - strong_low, 4) )

print("\nRandom Forest")
print("-" * 14)
print("Lowest AP:", round(rf_low, 4))
print("Highest AP:", round(rf_high, 4))
print("Spread:", round(rf_high - rf_low, 4))

print("\nGain over Strong Baseline")
print("-" * 27)
print("Lowest Gain", round(gain_low, 4))
print("Highest Gain", round(gain_high, 4))
print("Mean Gain", round(split_results_df["Gain vs Strong Baseline"].mean(), 4))

Strong Baseline
-----------------
Lowest AP: 0.2142
Highest AP: 0.2254
Spread: 0.0112

Random Forest
--------------
Lowest AP: 0.6104
Highest AP: 0.6157
Spread: 0.0053

Gain over Strong Baseline
---------------------------
Lowest Gain 0.3859
Highest Gain 0.3962
Mean Gain 0.3913


* **Spread:** Highest AP minus lowest AP for the same model.
* **Gain:** Random Forest AP minus Logistic Regression AP on the same split.
* If the Random Forest gain is positive on all three splits and is large relative to the score variation, that is encouraging evidence of improvement.
* If the gain changes sign or is small relative to the variation, the improvement is not yet convincing.

In [32]:
split_results_df.to_csv("Day42_split_variability.csv", index=False)

print("C3 results saved successfully")

C3 results saved successfully


# **Part D — The hierarchy of tuning effort**

* If Random Forest clearly wins: investigate feature importance and feature engineering tomorrow.
* If Logistic Regression wins: inspect the signal, possible data limitations, and whether the linear baseline is already sufficient.
* If both perform poorly: verify the target, features available before outreach, and the quality of the evaluation setup before tuning.

The current planned lever is feature engineering and checking the usefulness of available predictors. The lever deliberately not being pulled today is hyperparameter tuning, because today's objective is to establish an honest untuned result.  
If the first model loses to the strong baseline, tomorrow should begin with an error and feature analysis rather than immediately training another model.

# **Part E — Document the gap to goal**

In [34]:
gap_to_target = good_enough_target - model_ap
original_distance = good_enough_target - trivial_ap

if original_distance > 0:
    fraction_covered = ((model_ap - trivial_ap) / original_distance)
else:
    fraction_covered = np.nan


print("GOOD ENOUGH target:", round(good_enough_target, 4))
print("Current model AP:", round(model_ap, 4))
print("Gap (target - current):", round(gap_to_target, 4))

GOOD ENOUGH target: 0.2
Current model AP: 0.6104
Gap (target - current): -0.4104


In [35]:
if not np.isnan(fraction_covered):
    print("Original target distance covered:", round(fraction_covered * 100, 2),"%")

if model_ap >= good_enough_target:
    print("STATUS: GOOD ENOUGH target reached.")

else:
    print("STATUS: GOOD ENOUGH not reached yet.")

Original target distance covered: 594.41 %
STATUS: GOOD ENOUGH target reached.


* A positive gap means you still need to improve.
* A gap of zero means you've reached the target.
* A negative gap means you have exceeded the target.  

The percentage measures progress from the trivial baseline towards the target. It can be negative if the model performs worse than the trivial baseline, or exceed 100% if the target is surpassed.  
The conclusion about whether the target is reachable should be based on the actual AP scores, the strength of the baseline, and the cross-validation spread—not optimism alone.

# **Part F — Start the decision log**